In [ ]:
from icl.coin.analysis.interventions import (
    intervene_direct_injection_coin,
    plot_inject_posterior_per_position,
)
from icl.utils.unified_interface import get_exp_name
exp_name = get_exp_name("coin", k=-1)
result = intervene_direct_injection_coin(
    exp_name=exp_name,
    layer=4,
    B=64,
    n_samples=2**14,
    eval_positions=list(range(30)),
    fit_n_samples=2**13,
    fit_positions=list(range(1, 20)),
    center_task_vecs=True,
    dirichlet_alpha=1.0,
)
fig = plot_inject_posterior_per_position(result)

## $\lambda$–posterior agreement metrics

### Goal

Quantify how well the projected coefficients $\boldsymbol{\lambda}(t)$
(from the hidden-state projection onto task vectors) track the Bayesian
posterior $\mathbf{p}^*(t) = P(Z{=}k \mid x_{1:t})$ over time.

### Metrics (three panels)

**Total variation distance**:

$$
\mathrm{TV}(t) = \frac{1}{2}\sum_{k=1}^{K}
  \bigl|\lambda_k(t) - p^*_k(t)\bigr|,
$$

averaged over samples and tasks.  $\mathrm{TV} \in [0, 1]$; lower is better.


### Method

1. Fit the linear probe (same as in the trajectory projection) to obtain
   centred task vectors $\widetilde{W}_k$.
2. For each task (major and OOD), project per-sample hidden trajectories
   onto $\widetilde{W}_k$ via OLS and map to the simplex to get
   $\boldsymbol{\lambda}(t)$.
3. Compute the Bayesian posterior $\mathbf{p}^*(t)$ from the observed tokens.
4. Evaluate TV, cosine, and rolling correlation at each position $t$.

In [ ]:
from icl.coin.coin_analysis import plot_lambda_posterior_agreement_coin
from icl.utils.unified_interface import get_exp_name

exp_name = get_exp_name("coin", -1, vocab_size=VOCAB_SIZE, major_pool_type=POOL_TYPE, major_means=MAJOR_MEANS)
result = plot_lambda_posterior_agreement_coin(
    exp_name=exp_name,
    layer_index=5,
    n_ood=32,
    min_position=1,
    max_position=30,
    fit_n_samples=8000,
    fit_positions=list(range(128)),
    fit_include_position_bias=False,
    figsize=(8, 4),
    major_only=True,
)

In [ ]:
from icl.coin.coin_analysis import plot_lambda_posterior_agreement_coin
from icl.utils.unified_interface import get_exp_name

exp_name = get_exp_name("coin", -1, vocab_size=VOCAB_SIZE, major_pool_type=POOL_TYPE, major_means=MAJOR_MEANS)
result = plot_lambda_posterior_agreement_coin(
    exp_name=exp_name,
    layer_index=4,
    n_ood=32,
    min_position=1,
    max_position=30,
    fit_n_samples=8000,
    fit_positions=list(range(128)),
    fit_include_position_bias=False,
    figsize=(6, 4),
    major_only=True,
)

## Projected coefficients $\boldsymbol{\lambda}(t)$ vs Bayesian posterior

This plot compares the projection coefficients $\boldsymbol{\lambda}(t) = (\lambda_1(t), \ldots, \lambda_{K_\text{maj}}(t))$ extracted from the hidden states with the true Bayesian posterior $P(Z=k \mid x_{1:t})$.

**Computing $\boldsymbol{\lambda}(t)$:**

1. **Linear probe.** Fit a reverse linear probe in major mode at late positions: $\mathbf{h} \approx \boldsymbol{\pi} W + b$, where $W \in \mathbb{R}^{K_\text{maj} \times D}$ and $b \in \mathbb{R}^D$. Define centered task vectors $\tilde{W}_j = W_j - \bar{W}$.

2. **Hidden deviation.** For a single sequence sample at position $t$, compute $\mathbf{d}(t) = \mathbf{h}(t) - b - \bar{W}$. This subtracts both the fitted bias and the mean row of $W$, isolating the posterior-encoding component.

3. **Constrained least-squares.** Find $\boldsymbol{\lambda}(t)$ such that $\mathbf{d}(t) \approx \sum_j \lambda_j(t) \tilde{W}_j$ subject to $\sum_j \lambda_j = 1$. With $K_\text{maj}=3$, this is solved by dropping one task vector (since $\tilde{W}$ is zero-mean), solving the 2D least-squares $\hat{\mathbf{w}} = (X^\top X)^{-1} X^\top \mathbf{d}(t)$, and recovering $\lambda_3 = 1 - \lambda_1 - \lambda_2$.

4. **Simplex projection.** If $\boldsymbol{\lambda}$ falls outside the probability simplex, project it onto $\Delta^{K_\text{maj}-1}$ via Euclidean simplex projection.

**Ideal relationship:** If the hidden state perfectly encodes the posterior via the linear map $W$, then $\lambda_k(t) = P(Z=k \mid x_{1:t})$. Close agreement between the solid ($\lambda_k$) and dashed (true posterior) curves confirms this.

In [ ]:
from icl.coin.analysis import traj_cellmean_projection_plot_coin
from icl.utils.unified_interface import get_exp_name

exp_name = get_exp_name("coin", k=-1)
result = traj_cellmean_projection_plot_coin(exp_name, layer_index=3, annotate_agreement=False,)

In [ ]:
from icl.coin.coin_analysis import traj_post_posterior_projection_plot_coin
from icl.utils.unified_interface import get_exp_name

exp_name = get_exp_name("coin", -1, vocab_size=VOCAB_SIZE, major_pool_type=POOL_TYPE, major_means=MAJOR_MEANS)
result = traj_post_posterior_projection_plot_coin(
    exp_name=exp_name,
    layer_index=4,
    task_ids=range(1),
    fit_n_samples=5000,
    fit_positions=list(range(128)),
    fit_include_position_bias=False,
    B=32,
    annotate_agreement=False,
)